In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging
from pathlib import Path

from transformers import GenerationConfig

from pivotal_tokens.hf import load_model, load_tokenizer
from pivotal_tokens.extractor import SuccessProbabilityShiftExtractor
from pivotal_tokens.oracle import RegexOracle
from pivotal_tokens.repo import DictRepo
from pivotal_tokens.utils import setup_logging

/Users/maaxap/Workspace/Repos/phd/pivotal_tokens/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
HF_MODEL_ID = "Qwen/Qwen3-1.7B"
DICT_REPO_DIR = Path("repo")
ORACLE_ANSWER_REGEX = r"(?s)\s*(?:Answer:\s*)?(.*?)\s*(?=(?:<\|im_end\|>|<\|endoftext\|>|\Z))"

SYSTEM_PROMPT = ("Answer the following question directly, strictly without chain-of-thought after \"</think>\"."
                 "Keep the answer short (e.g., \"yes\" or \"no\" for binary questions, a person's full name if "
                 "the question expects a person, a country name if it asks about a country, etc.). Output the "
                 "answer strictly after the prefix \"Answer: \"  with no extra text.")

In [4]:
setup_logging(logging.DEBUG)

In [5]:
model = load_model(model_id=HF_MODEL_ID, device="cpu")

2025-12-07 00:50:49,129 - urllib3.connectionpool - DEBUG - Starting new HTTPS connection (1): huggingface.co:443
2025-12-07 00:52:09,457 - urllib3.connectionpool - DEBUG - https://huggingface.co:443 "HEAD /Qwen/Qwen3-1.7B/resolve/main/config.json HTTP/1.1" 307 0
2025-12-07 00:52:09,493 - urllib3.connectionpool - DEBUG - https://huggingface.co:443 "HEAD /api/resolve-cache/models/Qwen/Qwen3-1.7B/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e/config.json HTTP/1.1" 200 0
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  3.42it/s]
2025-12-07 00:52:10,342 - urllib3.connectionpool - DEBUG - https://huggingface.co:443 "HEAD /Qwen/Qwen3-1.7B/resolve/main/generation_config.json HTTP/1.1" 307 0
2025-12-07 00:52:10,372 - urllib3.connectionpool - DEBUG - https://huggingface.co:443 "HEAD /api/resolve-cache/models/Qwen/Qwen3-1.7B/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e/generation_config.json HTTP/1.1" 200 0
2025-12-07 00:52:10,545 - urllib3.connectionpool - DEBUG - https://huggingface.co:443 

In [6]:
tokenizer = load_tokenizer(model_id=HF_MODEL_ID)

2025-12-07 00:52:10,725 - urllib3.connectionpool - DEBUG - https://huggingface.co:443 "HEAD /Qwen/Qwen3-1.7B/resolve/main/tokenizer_config.json HTTP/1.1" 307 0
2025-12-07 00:52:10,761 - urllib3.connectionpool - DEBUG - https://huggingface.co:443 "HEAD /api/resolve-cache/models/Qwen/Qwen3-1.7B/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e/tokenizer_config.json HTTP/1.1" 200 0
2025-12-07 00:52:10,918 - urllib3.connectionpool - DEBUG - https://huggingface.co:443 "GET /api/models/Qwen/Qwen3-1.7B/tree/main/additional_chat_templates?recursive=False&expand=False HTTP/1.1" 404 64


In [7]:
base_repo = DictRepo(dirpath=DICT_REPO_DIR)
oracle = RegexOracle(answer_regex=ORACLE_ANSWER_REGEX,
                        fuzzy_match_threshold=0.5)

In [8]:
generation_config = GenerationConfig(temperature=0.6,
                                        top_p=0.95,
                                        top_k=20,
                                        min_p=0.1,
                                        max_new_tokens=32,
                                        do_sample=True)

In [9]:
extractor = SuccessProbabilityShiftExtractor(model=model,
                                                tokenizer=tokenizer,
                                                oracle=oracle,
                                                base_repo=base_repo,
                                                prob_threshold=0.05,
                                                num_trials=2,
                                                min_prob=0.1,
                                                max_prob=0.9,
                                                batch_size=2,
                                                generation_config=generation_config) 

In [10]:
# reasoning_trace = ("It is Prague")
reasoning_trace = ("The capital of Czech Republic is Paris. Wait, I'm wrong, it is Prague.")
expected_answer = "Prague"
user_prompt = "What is the capital of Czech Republic?"
metadata = {"example_id": "debug_001"}

In [11]:
!rm -rf repo

In [12]:
extractor.clear_cache()

pivotal_tokens = extractor.extract(reasoning_trace=reasoning_trace,
                                    system_prompt=SYSTEM_PROMPT,
                                    user_prompt=user_prompt,
                                    expected_answer=expected_answer,
                                    actual_answer="Prague",
                                    sample_id="example_1",
                                    metadata=metadata)

2025-12-07 00:52:11,451 - root - DEBUG - Starting extraction for sample example_1
2025-12-07 00:52:11,452 - root - DEBUG - Creating SampleRepo for sample ID: example_1
2025-12-07 00:52:11,452 - root - DEBUG - Saving sample metadata for sample ID: example_1
2025-12-07 00:52:11,453 - root - DEBUG - Estimating success probability for key: ('', 'Answer the following question directly, strictly without chain-of-thought after "</think>".Keep the answer short (e.g., "yes" or "no" for binary questions, a person\'s full name if the question expects a person, a country name if it asks about a country, etc.). Output the answer strictly after the prefix "Answer: "  with no extra text.', 'What is the capital of Czech Republic?', 2, 'Prague')
`generation_config` default values have been modified to match model-specific defaults: {'pad_token_id': 151643, 'bos_token_id': 151643, 'eos_token_id': [151645, 151643]}. If this is not desired, please set these values explicitly.
2025-12-07 00:52:18,276 - roo

In [13]:
print("Extracted Pivotal Tokens:", pivotal_tokens)

Extracted Pivotal Tokens: [SuccessProbabilityShiftSpan(token_ids=[151667, 198, 785, 6722, 315, 33150, 5429, 374, 12095, 13], span_text='<think>\nThe capital of Czech Republic is Paris.', prob_before=0.0, prob_after=0.0, prob_delta=0.0, pivotal_context='<|im_start|>system\nAnswer the following question directly, strictly without chain-of-thought after "</think>".Keep the answer short (e.g., "yes" or "no" for binary questions, a person\'s full name if the question expects a person, a country name if it asks about a country, etc.). Output the answer strictly after the prefix "Answer: "  with no extra text.<|im_end|>\n<|im_start|>user\nWhat is the capital of Czech Republic?<|im_end|>\n<|im_start|>assistant\n', metadata={'example_id': 'debug_001'}), SuccessProbabilityShiftSpan(token_ids=[13824], span_text=' Wait', prob_before=0.0, prob_after=1.0, prob_delta=1.0, pivotal_context='<|im_start|>system\nAnswer the following question directly, strictly without chain-of-thought after "</think>".Ke

In [14]:
for pt in pivotal_tokens:
    print(f"Span: '{pt.span_text}'")
    print("***")

Span: '<think>
The capital of Czech Republic is Paris.'
***
Span: ' Wait'
***
Span: ','
***
Span: ' I'm wrong'
***
Span: ', it is Prague.
</think>'
***
